In [ ]:
# ============================================================
# FINAL FIXED ONE-CELL CODE
# PatchCore Essential Edge Metrics
# Method: PatchCore + Reservoir Sampling + Top-k Mean
# Dataset: Lusitano_Dataset
# SEED = 42 | No CenterCrop
#
# Fixed using your working PaDiM structure:
#   1. Force-remount Google Drive
#   2. Robust rsync copy instead of shutil.copytree
#   3. Force CUDA GPU
#   4. NUM_WORKERS = 0 for Colab stability
#   5. Safe image verification/loading
#   6. Dataset count validation
#   7. AUC, AP/mAP, F1, confusion matrix, timing, memory
# ============================================================

# ============================================================
# 0) Imports
# ============================================================

import os
import gc
import time
import random
import shutil
import subprocess
import psutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
from PIL import Image, ImageFile

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from IPython.display import display
from google.colab import drive

# ============================================================
# 1) Safe Google Drive mount
# ============================================================

try:
    drive.flush_and_unmount()
    time.sleep(2)
except Exception:
    pass

drive.mount("/content/drive", force_remount=True)

# ============================================================
# 2) Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

# ============================================================
# 3) Force GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not active. Go to Runtime > Change runtime type > GPU, "
        "then restart the runtime and run again."
    )

DEVICE = "cuda"
print("Using device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 4) Dataset paths: robust Drive -> local /content copy
# ============================================================

COPY_DATASET_TO_LOCAL = True

DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset")
LOCAL_DATASET_ROOT = Path("/content/Lusitano_Dataset")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def count_images_fast(folder: Path):
    folder = Path(folder)

    if not folder.exists():
        return 0

    return sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

def robust_copy_drive_to_local(src: Path, dst: Path):
    """
    More stable than shutil.copytree for Google Drive in Colab.
    Useful for many .tif images.
    """

    if dst.exists():
        print("Removing incomplete/old local dataset copy...")
        shutil.rmtree(dst)

    print("Copying dataset from Google Drive to local Colab storage using rsync...")
    print("Source:", src)
    print("Target:", dst)

    copy_start = time.time()

    cmd = [
        "rsync",
        "-ah",
        "--info=progress2",
        "--partial",
        "--ignore-missing-args",
        f"{str(src)}/",
        f"{str(dst)}/",
    ]

    result = subprocess.run(cmd)

    if result.returncode != 0:
        raise RuntimeError(
            "Dataset copy failed. Google Drive likely disconnected. "
            "Please rerun the cell after Drive remounts successfully."
        )

    print(f"Dataset copy completed in {(time.time() - copy_start) / 60:.2f} minutes.")

if COPY_DATASET_TO_LOCAL:
    if not DRIVE_DATASET_ROOT.exists():
        raise ValueError(f"Drive dataset not found: {DRIVE_DATASET_ROOT}")

    source_count = count_images_fast(DRIVE_DATASET_ROOT)

    if source_count == 0:
        raise RuntimeError(
            f"No images found in Drive dataset root: {DRIVE_DATASET_ROOT}. "
            "Check dataset path or Drive permission."
        )

    expected_local_train = LOCAL_DATASET_ROOT / "nondefects" / "nondefects"
    expected_local_test  = LOCAL_DATASET_ROOT / "test" / "test"

    local_count = count_images_fast(LOCAL_DATASET_ROOT)

    print("Drive image count:", source_count)
    print("Existing local image count:", local_count)

    local_copy_ok = (
        expected_local_train.exists()
        and expected_local_test.exists()
        and local_count >= source_count * 0.98
    )

    if local_copy_ok:
        print("Local dataset already exists and looks complete:", LOCAL_DATASET_ROOT)
    else:
        robust_copy_drive_to_local(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)

        local_count = count_images_fast(LOCAL_DATASET_ROOT)
        print("Final local image count:", local_count)

        if local_count < source_count * 0.98:
            raise RuntimeError(
                f"Local dataset copy may be incomplete. "
                f"Drive images: {source_count}, local images: {local_count}."
            )

    DATASET_ROOT = LOCAL_DATASET_ROOT

else:
    DATASET_ROOT = DRIVE_DATASET_ROOT

print("Using DATASET_ROOT:", DATASET_ROOT)

train_good_path = DATASET_ROOT / "nondefects" / "nondefects"
test_root_path  = DATASET_ROOT / "test" / "test"

if not train_good_path.exists():
    raise ValueError(f"Training path not found: {train_good_path}")

if not test_root_path.exists():
    raise ValueError(f"Test path not found: {test_root_path}")

print("Train folder:", train_good_path)
print("Test folder :", test_root_path)

# ============================================================
# 5) Fixed experiment settings
# ============================================================

METHOD_NAME = "PatchCore + Reservoir Sampling + Top-k Mean"

IMG_SIZE = 448

# Safer than 16 for Colab. You can increase to 16 after successful run.
BATCH_SIZE = 8

# Keep 0 for Colab stability.
NUM_WORKERS = 0

PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
MAX_MEM_PATCHES = 20_000

NN_CHUNK = 40_000
THRESH_SAMPLE_IMAGES = 2000

SCORE_MODE = "topk_mean"
TOPK_FRAC = 0.01

SAVE_DIR = Path("/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/localcopy_compute_timing_fixed_patchcore")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = SAVE_DIR / "patchcore_reservoir_topk_mean_fixed_seed42.csv"
SCORES_CSV = SAVE_DIR / "patchcore_reservoir_topk_mean_image_scores_seed42.csv"
BAD_IMAGES_CSV = SAVE_DIR / "patchcore_skipped_bad_images_seed42.csv"

# ============================================================
# 6) Metric / memory helpers
# ============================================================

def bytes_to_mb(x):
    return x / (1024 ** 2)

def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())

def model_size_mb(model):
    total_bytes = 0

    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()

    for b in model.buffers():
        total_bytes += b.numel() * b.element_size()

    return bytes_to_mb(total_bytes)

def current_ram_mb():
    process = psutil.Process(os.getpid())
    return bytes_to_mb(process.memory_info().rss)

def reset_peak_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

def get_peak_memory_mb():
    torch.cuda.synchronize()
    return bytes_to_mb(torch.cuda.max_memory_allocated())

def edge_efficiency_score(auc, ap, f1, time_per_image, total_mb, peak_gpu_mb):
    numerator = 0.25 * auc + 0.35 * ap + 0.40 * f1
    denominator = 0.20 * time_per_image + 0.40 * total_mb + 0.40 * peak_gpu_mb

    if denominator <= 0:
        return 0.0

    return numerator / denominator

# ============================================================
# 7) Transform
# No CenterCrop is used
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# ============================================================
# 8) Robust image listing and datasets
# ============================================================

bad_images = []

def is_readable_image(path: Path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception as e:
        bad_images.append({
            "image_path": str(path),
            "error": repr(e),
        })
        return False

def list_images_safe(folder: Path, recursive=True, verify_images=True):
    folder = Path(folder)

    if recursive:
        iterator = sorted(folder.rglob("*"))
    else:
        iterator = sorted(folder.glob("*"))

    paths = []

    for p in iterator:
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            if verify_images:
                if is_readable_image(p):
                    paths.append(p)
            else:
                paths.append(p)

    return paths

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]

        try:
            img = Image.open(p).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to open image: {p} | Error: {repr(e)}")

        return self.transform(img), str(p)

class TestImageDataset(Dataset):
    def __init__(self, items, transform):
        self.items = list(items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        p, label = self.items[idx]

        try:
            img = Image.open(p).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed to open image: {p} | Error: {repr(e)}")

        return self.transform(img), int(label), str(p)

# ============================================================
# 9) Feature extractor: EfficientNet-B5
# ============================================================

class EfficientNetFeatureExtractor(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        self.model.eval()

        for p in self.model.parameters():
            p.requires_grad = False

        self.features = []

        def hook(_, __, output):
            self.features.append(output)

        self.model.features[3].register_forward_hook(hook)
        self.model.features[5].register_forward_hook(hook)
        self.model.features[7].register_forward_hook(hook)

    @torch.no_grad()
    def forward(self, x):
        self.features = []

        _ = self.model(x)

        if len(self.features) != 3:
            raise RuntimeError(f"Expected 3 feature maps, got {len(self.features)}.")

        fmap_size = min(f.shape[-2] for f in self.features)
        resize = torch.nn.AdaptiveAvgPool2d(fmap_size)

        resized = [resize(f) for f in self.features]
        patch_features = torch.cat(resized, dim=1)

        B, C, H, W = patch_features.shape
        patch_features = patch_features.reshape(B, C, H * W).permute(0, 2, 1)

        return patch_features

# ============================================================
# 10) Load training images and test images
# ============================================================

print("\nScanning training images...")
train_paths = list_images_safe(train_good_path, recursive=True, verify_images=True)

if len(train_paths) == 0:
    raise ValueError("No valid training images found.")

print("Valid training normal images:", len(train_paths))

def get_label(folder_name):
    name = folder_name.lower().replace("_", "-").strip()

    if name == "non-defects":
        return 0

    if name == "defects":
        return 1

    return None

test_items = []

print("\nScanning test images...")

for folder in sorted(test_root_path.iterdir()):
    if not folder.is_dir():
        continue

    label = get_label(folder.name)

    if label is None:
        print("Skipping unknown folder:", folder.name)
        continue

    paths = list_images_safe(folder, recursive=True, verify_images=True)

    print(folder.name, "valid images:", len(paths))

    for p in paths:
        test_items.append((p, label))

if len(bad_images) > 0:
    bad_df = pd.DataFrame(bad_images)
    bad_df.to_csv(BAD_IMAGES_CSV, index=False)

    print("\nSkipped unreadable images:", len(bad_images))
    print("Saved bad image list:", BAD_IMAGES_CSV)

if len(test_items) == 0:
    raise ValueError("No valid test images found.")

normal_count = sum(1 for _, y in test_items if y == 0)
defect_count = sum(1 for _, y in test_items if y == 1)

print("\nTotal valid test images:", len(test_items))
print("Valid normal test images:", normal_count)
print("Valid defect test images:", defect_count)

if normal_count < 900:
    raise RuntimeError(
        f"Normal test count is too low: {normal_count}. "
        "Expected around 1038 for Lusitano. "
        "Your local dataset copy may be incomplete or folder structure may be wrong."
    )

if defect_count < 1500:
    raise RuntimeError(
        f"Defect test count is too low: {defect_count}. "
        "Expected around 1646 for Lusitano."
    )

train_loader = DataLoader(
    ImagePathDataset(train_paths, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

# ============================================================
# 11) Initialize backbone
# ============================================================

backbone = EfficientNetFeatureExtractor().to(DEVICE).eval()
backbone_size_mb = model_size_mb(backbone)

print(f"\nBackbone/model size MB: {backbone_size_mb:.2f}")

# ============================================================
# 12) Build common candidate patch pool
# ============================================================

set_seed(SEED)

all_features = []

print("\nExtracting candidate patch pool...")

for xb, _ in tqdm(train_loader, desc="Candidate pool"):
    xb = xb.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        feats = backbone(xb)

    B, N, C = feats.shape

    for i in range(B):
        n = min(PATCHES_PER_IMAGE, N)
        idx = torch.randperm(N, device=DEVICE)[:n]
        sampled = feats[i, idx].detach().float().cpu()
        all_features.append(sampled)

    del xb, feats
    gc.collect()
    torch.cuda.empty_cache()

candidate_pool = torch.cat(all_features, dim=0)
del all_features
gc.collect()

print("Candidate pool before PRE_POOL cap:", tuple(candidate_pool.shape))

if candidate_pool.shape[0] > PRE_POOL:
    set_seed(SEED)
    idx = torch.randperm(candidate_pool.shape[0])[:PRE_POOL]
    candidate_pool = candidate_pool[idx].contiguous()
    gc.collect()

print("Final candidate pool:", tuple(candidate_pool.shape))

# ============================================================
# 13) Memory-bank selection: Reservoir Sampling
# ============================================================

def reservoir_sample(features_cpu, max_samples, seed=42):
    random.seed(seed)

    N, C = features_cpu.shape

    if N <= max_samples:
        return features_cpu.clone()

    memory_bank = torch.empty((max_samples, C), dtype=torch.float32)

    for i in tqdm(range(N), desc="Reservoir sampling"):
        if i < max_samples:
            memory_bank[i] = features_cpu[i]
        else:
            j = random.randint(0, i)
            if j < max_samples:
                memory_bank[j] = features_cpu[i]

    return memory_bank

print("\nBuilding reservoir memory bank...")

memory_bank_cpu = reservoir_sample(
    candidate_pool,
    max_samples=MAX_MEM_PATCHES,
    seed=SEED,
)

del candidate_pool
gc.collect()

# ============================================================
# 14) Model / memory footprint
# ============================================================

memory_bank_size_mb = tensor_size_mb(memory_bank_cpu)
estimated_total_footprint_mb = memory_bank_size_mb + backbone_size_mb

print(f"\nMemory-bank size MB: {memory_bank_size_mb:.2f}")
print(f"Backbone/model size MB: {backbone_size_mb:.2f}")
print(f"Estimated total footprint MB: {estimated_total_footprint_mb:.2f}")

memory_bank_gpu = memory_bank_cpu.to(DEVICE, non_blocking=True)

# ============================================================
# 15) Image-level anomaly score: Top-k Mean
# Patch score = nearest-neighbor distance to memory bank
# Image score = mean of top 1% largest nearest-neighbor patch distances
# ============================================================

@torch.no_grad()
def image_anomaly_score(patch_feats_gpu, memory_bank_gpu, chunk_size=40_000):
    x = patch_feats_gpu.float()
    P = x.shape[0]

    min_dist = torch.full(
        (P,),
        float("inf"),
        device=DEVICE,
        dtype=torch.float32,
    )

    x2 = x.pow(2).sum(dim=1, keepdim=True)

    for start in range(0, memory_bank_gpu.shape[0], chunk_size):
        mb = memory_bank_gpu[start:start + chunk_size].float()
        mb2 = mb.pow(2).sum(dim=1).unsqueeze(0)

        dot = x @ mb.t()
        d2 = x2 + mb2 - 2.0 * dot
        d2 = torch.clamp(d2, min=0.0)

        min_dist = torch.minimum(min_dist, d2.min(dim=1).values)

    patch_scores = min_dist.sqrt()

    if SCORE_MODE == "max":
        return patch_scores.max().item()

    elif SCORE_MODE == "topk_mean":
        k = max(1, int(patch_scores.numel() * TOPK_FRAC))
        topk_scores = torch.topk(patch_scores, k=k, largest=True).values
        return topk_scores.mean().item()

    else:
        raise ValueError(f"Unknown SCORE_MODE: {SCORE_MODE}")

# ============================================================
# 16) Threshold for F1 calculation
# ============================================================

rng_thresh = np.random.default_rng(SEED + 100)

num_for_thresh = min(THRESH_SAMPLE_IMAGES, len(train_paths))
thresh_indices = rng_thresh.permutation(len(train_paths))[:num_for_thresh]
thresh_subset = [train_paths[i] for i in thresh_indices]

thresh_loader = DataLoader(
    ImagePathDataset(thresh_subset, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

threshold_scores = []

print("\nComputing threshold from normal training subset...")

for xb, _ in tqdm(thresh_loader, desc="Threshold"):
    xb = xb.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        feats = backbone(xb)

    for i in range(feats.shape[0]):
        score = image_anomaly_score(
            feats[i],
            memory_bank_gpu,
            chunk_size=NN_CHUNK,
        )
        threshold_scores.append(score)

    del xb, feats
    gc.collect()
    torch.cuda.empty_cache()

threshold_scores = np.array(threshold_scores)

threshold = threshold_scores.mean() + 3.0 * threshold_scores.std(ddof=1)

print(f"Internal threshold for F1: {threshold:.6f}")

# ============================================================
# 17) Test evaluation + inference time + peak memory usage
# ============================================================

test_loader = DataLoader(
    TestImageDataset(test_items, transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
)

reset_peak_memory()

torch.cuda.synchronize()

end_to_end_start = time.perf_counter()
compute_total_time = 0.0

y_true = []
y_score = []
image_paths = []

print("\nEvaluating test set...")

for xb, yb, paths in tqdm(test_loader, desc="Testing"):
    torch.cuda.synchronize()
    compute_start = time.perf_counter()

    xb = xb.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        feats = backbone(xb)

    for i in range(feats.shape[0]):
        score = image_anomaly_score(
            feats[i],
            memory_bank_gpu,
            chunk_size=NN_CHUNK,
        )

        y_score.append(score)
        y_true.append(int(yb[i]))
        image_paths.append(str(paths[i]))

    torch.cuda.synchronize()
    compute_total_time += time.perf_counter() - compute_start

    del xb, feats
    gc.collect()
    torch.cuda.empty_cache()

torch.cuda.synchronize()

end_to_end_total_time = time.perf_counter() - end_to_end_start

compute_inference_time_per_image = compute_total_time / len(y_true)
end_to_end_local_runtime_per_image = end_to_end_total_time / len(y_true)
peak_memory_usage_mb = get_peak_memory_mb()

# ============================================================
# 18) Metrics
# ============================================================

y_true = np.array(y_true)
y_score = np.array(y_score)

y_pred = (y_score > threshold).astype(int)

auc_roc = roc_auc_score(y_true, y_score)
map_ap = average_precision_score(y_true, y_score)
f1 = f1_score(y_true, y_pred)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

edge_eff = edge_efficiency_score(
    auc=auc_roc,
    ap=map_ap,
    f1=f1,
    time_per_image=compute_inference_time_per_image,
    total_mb=estimated_total_footprint_mb,
    peak_gpu_mb=peak_memory_usage_mb,
)

# ============================================================
# 19) Final result table
# ============================================================

result = {
    "Method": METHOD_NAME,
    "Seed": SEED,
    "Dataset_Root_Used": str(DATASET_ROOT),
    "IMG_SIZE": IMG_SIZE,
    "BATCH_SIZE": BATCH_SIZE,
    "NUM_WORKERS": NUM_WORKERS,
    "Score_Mode": SCORE_MODE,
    "TopK_Frac": TOPK_FRAC,
    "PATCHES_PER_IMAGE": PATCHES_PER_IMAGE,
    "PRE_POOL": PRE_POOL,
    "MAX_MEM_PATCHES": MAX_MEM_PATCHES,
    "NN_CHUNK": NN_CHUNK,
    "Internal_Threshold": threshold,
    "AUC_ROC": auc_roc,
    "mAP_AP": map_ap,
    "F1_Score": f1,
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
    "Inference_Time_Per_Image_sec": compute_inference_time_per_image,
    "Compute_Inference_Time_Per_Image_sec": compute_inference_time_per_image,
    "End_To_End_Local_Runtime_Per_Image_sec": end_to_end_local_runtime_per_image,
    "Compute_Total_Time_sec": compute_total_time,
    "End_To_End_Local_Total_Time_sec": end_to_end_total_time,
    "Memory_Bank_Size_MB": memory_bank_size_mb,
    "Backbone_Model_Size_MB": backbone_size_mb,
    "Estimated_Total_Footprint_MB": estimated_total_footprint_mb,
    "Peak_Memory_Usage_MB": peak_memory_usage_mb,
    "Edge_Efficiency": edge_eff,
    "Train_Normal_Images": len(train_paths),
    "Test_Normal_Images": normal_count,
    "Test_Defect_Images": defect_count,
    "Total_Test_Images": len(test_items),
    "Skipped_Bad_Images": len(bad_images),
}

df_result = pd.DataFrame([result])

df_scores = pd.DataFrame({
    "image_path": image_paths,
    "gt_label": y_true,
    "patchcore_score": y_score,
    "pred_label": y_pred,
})

df_result.to_csv(RESULT_CSV, index=False)
df_scores.to_csv(SCORES_CSV, index=False)

print("\n==============================")
print("FIXED PATCHCORE RESERVOIR TOP-K MEAN RESULT")
print("==============================")
display(df_result)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Defect"],
    digits=4,
))

print("\nSaved result:")
print(RESULT_CSV)

print("\nSaved image scores:")
print(SCORES_CSV)

if len(bad_images) > 0:
    print("\nSaved skipped bad images:")
    print(BAD_IMAGES_CSV)